In [2]:
import pandas as pd
import numpy as np
import string
import pickle
import spacy
import os

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from gensim.models import Word2Vec


In [3]:
def get_tag(tag):
    tag = tag.lower()
    if tag.startswith('j'): return 'a'
    if tag.startswith('v'): return 'v'
    if tag.startswith('n'): return 'n'
    if tag.startswith('r'): return 'r'
    return 'n'

def lemma(removed_stop):
    lemmatizer = WordNetLemmatizer()
    lemmatize = []
    tagged = pos_tag(removed_stop)

    for word, tag in tagged:
        label = get_tag(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemmatize.append(result)

    return lemmatize

def preprocessing(sentence):
    sentence = sentence.lower()
    punct = string.punctuation
    stop = stopwords.words('english')

    word_list = word_tokenize(sentence)
    word_list = [word for word in word_list if word.isalpha()]

    removed_punct = [token for token in word_list if token not in punct]
    removed_stop = [token for token in removed_punct if token not in stop]
    lemmatized = lemma(removed_stop)

    return lemmatized

nlp = spacy.load("en_core_web_sm")
NER_TYPES = ['ORG', 'PERSON', 'GPE', 'NORP']

def perform_ner(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities

def get_entity_dictionary(text):
    doc = nlp(text)
    ent_dict = {t: [] for t in NER_TYPES}
    for ent in doc.ents:
        if ent.label_ in NER_TYPES:
            token = ent.text.strip().lower().replace(" ", "_")
            ent_dict[ent.label_].append(token)
    return {k: " ".join(v) for k, v in ent_dict.items()}


In [4]:
data = pd.read_csv("Political_Bias_Cleaned_Balanced.csv")
data['Title'] = [title + " " + source for (title, source) in zip(data['Title'], data['Source'])]

dropped = ['id', 'Text', 'Link', 'Source']
data = data.drop(dropped, axis=1)

data["cleaned_tokens"] = data["Title"].apply(preprocessing)
data['entity_dict'] = data['Title'].apply(get_entity_dictionary)


In [5]:
print(data['cleaned_tokens'])
print(data['entity_dict'])

0       [john, bolton, advice, turmoil, trump, adminis...
1       [shrubland, make, southern, california, wildfi...
2       [los, angeles, destruction, fuel, bad, policy,...
3       [acquire, greenland, good, idea, threaten, for...
4       [assess, trump, expansionist, design, thedispa...
                              ...                        
1900     [trump, slap, colombia, trade, travel, usatoday]
1901    [rain, arrive, california, fear, turn, mudslid...
1902     [wildfire, threaten, city, ever, risk, usatoday]
1903    [trump, order, strip, air, force, curriculum, ...
1904    [police, arrest, florida, man, allegedly, thre...
Name: cleaned_tokens, Length: 1905, dtype: object
0       {'ORG': 'the_'continuing_turmoil'', 'PERSON': ...
1       {'ORG': 'wildfires_inevitable_thedispatch', 'P...
2       {'ORG': '', 'PERSON': '', 'GPE': 'los_angeles'...
3       {'ORG': 'thedispatch', 'PERSON': '', 'GPE': 'g...
4        {'ORG': '', 'PERSON': '', 'GPE': '', 'NORP': ''}
                      

In [79]:
NER_FEATURE_BUDGET = {
    'ORG':    300,
    'PERSON': 200,
    'GPE':    150,
    'NORP':   200,
}

ner_vectorizers = {}

def get_entity_features():
    from sklearn.feature_extraction.text import TfidfVectorizer

    entity_vectorizer = TfidfVectorizer(min_df=2, max_features=1000)

    entity_features = entity_vectorizer.fit_transform(data['entity_strings'])

    return entity_features

def ner_features(fit=True):
    parts = []
    for ner_type, max_feat in NER_FEATURE_BUDGET.items():
        strings = data['entity_dict'].apply(lambda d: d[ner_type])
        if fit:
            vec = TfidfVectorizer(min_df=2, max_features=max_feat)
            mat = vec.fit_transform(strings).toarray()
            ner_vectorizers[ner_type] = vec
        else:
            vec = ner_vectorizers[ner_type]
            mat = vec.transform(strings).toarray()
        parts.append(mat)
        print(f"  {ner_type}: {mat.shape[1]} features")
    return np.hstack(parts)

In [80]:
w2v_model = Word2Vec(sentences=data['cleaned_tokens'], vector_size=100, window=5, min_count=1, workers=4)

def get_mean_vector(tokens):
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if not vectors:
        return np.zeros(100)
    return np.mean(vectors, axis=0)


In [81]:
class NGramLanguageModel:
    def __init__(self, n1, n2):
        self.n1 = n1
        self.n2 = n2
        self.vectorizer = TfidfVectorizer(ngram_range=(n1, n2), max_features=3000)

    def fit_transform(self, corpus):
        return self.vectorizer.fit_transform(corpus)

    def transform(self, corpus):
        return self.vectorizer.transform(corpus)

ngram_model = NGramLanguageModel(n1=1, n2=2)
ngram_features = ngram_model.fit_transform(data['Title'])

In [83]:
le = LabelEncoder()

def train_model():
    w2v_features = np.array(data['cleaned_tokens'].apply(get_mean_vector).tolist())
    tfidf_features = ngram_features.toarray()
    all_ner_feats = ner_features(fit=True)

    # X = np.hstack((w2v_features, tfidf_features, ner_features))
    X = np.hstack((w2v_features, tfidf_features, all_ner_feats))
    y = le.fit_transform(data['Bias'])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y
    )

    # ComplementNB harus positif
    X_train_pos = X_train - X_train.min(axis=0)
    X_test_pos = X_test - X_train.min(axis=0)

    model = ComplementNB()
    model.fit(X_train_pos, y_train)

    y_pred = model.predict(X_test_pos)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.2f}")

    with open('model.pkl', 'wb') as f:
        pickle.dump(model, f)

    return model, X_train.min(axis=0), y_test, X_test_pos

model, feat_min, y_test, X_test_pos = train_model()


  ORG: 138 features
  PERSON: 133 features
  GPE: 58 features
  NORP: 20 features
Accuracy: 0.88


In [84]:
def get_performance():
    if(os.path.exists("model.pkl")):
        with open('model.pkl', 'rb') as file:
            model = pickle.load(file)
            y_pred = model.predict(X_test_pos)

            print(classification_report(y_pred=y_pred, y_true=y_test))

            y_prob = model.predict_proba(X_test_pos)
            print(f"ROC AUC: {roc_auc_score(y_test, y_prob, multi_class='ovr'):.4f}")

get_performance()

              precision    recall  f1-score   support

           0       0.91      0.94      0.93        54
           1       0.87      0.88      0.87        81
           2       0.83      0.71      0.76        41
           3       0.86      0.92      0.89       103
           4       0.92      0.88      0.90       102

    accuracy                           0.88       381
   macro avg       0.88      0.87      0.87       381
weighted avg       0.88      0.88      0.88       381

ROC AUC: 0.9828


In [86]:
w2v_all   = np.array(data['cleaned_tokens'].apply(get_mean_vector).tolist())
tfidf_all = ngram_features.toarray()
ner_all   = ner_features(fit=False)

article_matrix = np.hstack((w2v_all, tfidf_all, ner_all))
print(f"Article matrix shape: {article_matrix.shape}")


def recommend(user_title, top_n=5):
    tokens   = preprocessing(user_title)
    w2v_vec  = get_mean_vector(tokens).reshape(1, -1)
    tfidf_vec = ngram_model.transform([user_title]).toarray()
    ner_vec  = ner_features(user_title)

    user_vec = np.hstack((w2v_vec, tfidf_vec, ner_vec))
    scores   = cosine_similarity(user_vec, article_matrix)[0]

    top_indices = scores.argsort()[::-1]
    results = []
    for idx in top_indices:
        title = data['Title'].iloc[idx]
        if title.strip().lower() == user_title.strip().lower():
            continue
        results.append((title, round(float(scores[idx]), 4)))
        if len(results) == top_n:
            break
    return results

  ORG: 138 features
  PERSON: 133 features
  GPE: 58 features
  NORP: 20 features
Article matrix shape: (1905, 3449)


In [87]:
def show_ner(title):
    entities = perform_ner(title)
    if not entities:
        print("No named entities found.")
        return
    print(f"\nNamed Entities in: '{title}'")
    for text, label in entities:
        print(f"  {text} -> {label}")

def main():
    current_title = "No Title"
    current_category = "Unknown"

    while True:
        print("\n")
        print("Political Bias Article Recommendation")
        print(f"Your Title    : {current_title}")
        print(f"Bias Category : {current_category}")
        print("1. Input an article title")
        print("2. View Article Recommendations")
        print("3. View NER")
        print("4. Exit")

        choice = input("  >> ").strip()

        if choice == "1":
            current_title = input("\nEnter article title: ")
            source = input("\nEnter article source: ")
            current_title = current_title + " " + source
            if len(current_title.split(' ')) > 3:
                tokens  = preprocessing(current_title)
                w2v_vec = get_mean_vector(tokens).reshape(1, -1)
                tfidf_v = ngram_model.transform([current_title]).toarray()
                ner_lab = perform_ner(current_title)
                ner_vec = get_ner_type_vector(ner_lab).reshape(1, -1)
                user_vec = np.hstack((w2v_vec, tfidf_v, ner_vec))

                user_vec_pos = user_vec - feat_min

                pred = model.predict(user_vec_pos)
                current_category = le.inverse_transform(pred)[0]
                print(f"\nPredicted bias: {current_category}")
            else:
                print("Please input a title more than 3 words!")

        elif choice == "2":
            if current_title == "No Title":
                print("\nPlease enter a title first!!")
            else:
                recs = recommend(current_title, top_n=5)
                print(f"\nTop 5 Recommendations For You:")
                for i, (title, score) in enumerate(recs, 1):
                    print(f" {i}. {title}  (similarity: {score})")

        elif choice == "3":
            if current_title == "No Title":
                print("\nPlease enter a title first (option 1).")
            else:
                show_ner(current_title)

        elif choice == "4":
            print("\nbye!")
            break
        else:
            print("\nInvalid choice. Please enter 1–4.")

main()




Political Bias Article Recommendation
Your Title    : No Title
Bias Category : Unknown
1. Input an article title
2. View Article Recommendations
3. View NER
4. Exit


KeyboardInterrupt: Interrupted by user